# Fine-Tuning

In [1]:
data = [
  "He composes songs and practices piano daily.",
  "She reads books and explores the nearby caves."
]

In [2]:
VOCAB_SIZE = 70
CONTEXT_LEN = 6
EMB_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [3]:
import os
import re
import torch
import json
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [4]:
save_dir = "models"

In [5]:
_ = torch.manual_seed(123)

In [6]:
def separate_dots(s):
    s = re.sub(r"\.", " . ", s)
    return s.strip()

text_data = [separate_dots(x) for x in data]

In [7]:
file_name = "vocab.json"
with open(os.path.join(save_dir, file_name)) as f:
    vocab = json.load(f)

In [8]:
# Display token:word mappings
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: .
1: He
2: She
3: and
4: art
5: books
6: builds
7: caves
8: climbs
9: collaborates
10: complex
11: composes
12: creative
13: curates
14: daily
15: designs
16: digital
17: documents
18: every
19: everyday
20: exhibitions
21: experiments
22: explores
23: fairs
24: filmmakers
25: for
26: friends
27: harmonies
28: her
29: in
30: jewelry
31: local
32: maps
33: marathons
34: models
35: mountains
36: music
37: navigation
38: nearby
39: newspaper
40: novel
41: novels
42: organizes
43: participates
44: photography
45: piano
46: practices
47: projects
48: puzzles
49: reads
50: regularly
51: rhythms
52: science
53: small
54: solves
55: songs
56: soundtracks
57: stars
58: studies
59: teaches
60: the
61: trains
62: trips
63: tunes
64: using
65: weekend
66: wildlife
67: with
68: wooden
69: writes


In [9]:
# build word -> index mapping
word_to_id = {word: idx for idx, word in enumerate(vocab)}

# build id -> word mapping
id_to_word = {idx: word for idx, word in enumerate(vocab)}

In [10]:
def text_to_token_ids(text, word_to_id):
    tokens = text.split()
    return [word_to_id[t] for t in tokens]

def token_ids_to_text(token_ids, id_to_word):
    words = [id_to_word[id] for id in token_ids]
    return " ".join(words)

In [11]:
class LLMDataset(Dataset):
    def __init__(self, texts, word_to_id, max_len):
        self.texts = texts
        self.word_to_id = word_to_id
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token_ids(self.texts[idx], self.word_to_id)[:self.max_len + 1])
        x = tokens[:-1]
        y = tokens[1:]
        return x, y

# create dataset + dataloader
train_dataset = LLMDataset(
    text_data,
    word_to_id,
    CONTEXT_LEN
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(len(train_dataloader))

1


In [12]:
class Attention(nn.Module):
    def __init__(self, d_in, d_out, context_length):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.register_buffer("mask",torch.triu(torch.ones(context_length, context_length),diagonal=1))

    def forward(self, x, return_weights=False):
        _, num_tokens, _ = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()

        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / (self.d_out ** 0.5),dim=-1)

        context_vec = attn_weights @ values

        if return_weights:
            return context_vec, attn_weights

        return context_vec

In [13]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.att = Attention(
            d_in=EMB_DIM,
            d_out=EMB_DIM,
            context_length=CONTEXT_LEN)

    def forward(self, x):
        shortcut = x
        x = self.att(x)
        x = x + shortcut
        return x

In [14]:
class GPTModel(nn.Module):
    def __init__(self):

        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMB_DIM)
        self.trf_block1 = TransformerBlock()
        self.trf_block2 = TransformerBlock()
        self.trf_block3 = TransformerBlock()
        self.trf_block4 = TransformerBlock()
        self.trf_block5 = TransformerBlock()
        self.trf_block6 = TransformerBlock()
        self.output_layer = nn.Linear(EMB_DIM, VOCAB_SIZE, bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(torch.arange(seq_len))

        x = tok_embeds + pos_embeds

        x = self.trf_block1(x)
        x = self.trf_block2(x)
        x = self.trf_block3(x)
        x = self.trf_block4(x)
        x = self.trf_block5(x)
        x = self.trf_block6(x)

        logits = self.output_layer(x)
        return logits

In [15]:
model = GPTModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.1)
loss_fn = nn.CrossEntropyLoss()

In [16]:
file_name = "parameters.bin"
model.load_state_dict(torch.load(os.path.join(save_dir, file_name)))

<All keys matched successfully>

In [17]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):

        logits = model(X)
        loss = loss_fn(logits.flatten(0, 1), y.flatten())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"batch: {batch + 1} loss: {loss:>7f}")

In [18]:
for epoch in range(EPOCHS):
    print("-------------------------------")
    print(f"Epoch: {epoch+1}")
    train(train_dataloader, model, loss_fn, optimizer)
    print("-------------------------------")
print("Done!")

-------------------------------
Epoch: 1
batch: 1 loss: 3.807468
-------------------------------
-------------------------------
Epoch: 2
batch: 1 loss: 3.444600
-------------------------------
-------------------------------
Epoch: 3
batch: 1 loss: 3.269085
-------------------------------
-------------------------------
Epoch: 4
batch: 1 loss: 3.061651
-------------------------------
-------------------------------
Epoch: 5
batch: 1 loss: 2.862042
-------------------------------
-------------------------------
Epoch: 6
batch: 1 loss: 2.712401
-------------------------------
-------------------------------
Epoch: 7
batch: 1 loss: 2.571401
-------------------------------
-------------------------------
Epoch: 8
batch: 1 loss: 2.423693
-------------------------------
-------------------------------
Epoch: 9
batch: 1 loss: 2.278619
-------------------------------
-------------------------------
Epoch: 10
batch: 1 loss: 2.145434
-------------------------------
-----------------------------

In [19]:
_ = model.eval()

In [20]:
def generate(start_text):
    token_ids = text_to_token_ids(start_text, word_to_id)
    num_new_tokens = CONTEXT_LEN - len(token_ids)
    idx = torch.tensor(token_ids).unsqueeze(0)

    for _ in range(num_new_tokens):
        idx_cond = idx[:, -CONTEXT_LEN:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    idx = idx.view(-1).tolist()
    text = token_ids_to_text(idx, id_to_word)
    return text

In [21]:
text = generate("He composes")
print("Generated text:", text)

Generated text: He composes songs and practices piano


In [22]:
text = generate("She reads")
print("Generated text:", text)

Generated text: She reads books and explores the


In [23]:
file_name = "updated-parameters.bin"
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_dir, file_name))